In [2]:

#   /content/drive/MyDrive/GNN_PROJECT/inputs/drug_list.csv
#   /content/drive/MyDrive/GNN_PROJECT/raw/full_database.xml

# %% 0. Install, mount Drive, imports, reproducibility
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch-geometric", "rdkit", "lxml"])

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from itertools import combinations
from io import StringIO
import copy, io, random, time
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from lxml import etree
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, brier_score_loss
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.data import Data, HeteroData
from torch_geometric.nn import SAGEConv, HeteroConv

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

PROJECT = Path("/content/drive/MyDrive/GNN_PROJECT")
INPUT = PROJECT / "inputs"; RAW = PROJECT / "raw"; OUT = PROJECT / "outputs_clean"
OUT.mkdir(parents=True, exist_ok=True)
CSV_FILE = INPUT / "drug_list.csv"
XML_FILE = RAW / "full_database.xml"
assert CSV_FILE.exists() and XML_FILE.exists(), "Check Google Drive paths."
NS = "{http://www.drugbank.ca}"

# %% 1. Extract selected drugs, all known DDI positives, targets and enzymes
drug_list = pd.read_csv(CSV_FILE)
id_column = next((c for c in ["drug_id", "drugbank_id", "drug_name", "name"] if c in drug_list.columns), None)
if id_column is None:
    raise ValueError(f"No drug ID/name column found: {drug_list.columns.tolist()}")
requested = set(drug_list[id_column].dropna().astype(str).str.strip().str.lower())

drugs, ddi_rows, target_rows, enzyme_rows, pubchem_rows = [], [], [], [], []
context = etree.iterparse(str(XML_FILE), events=("end",), tag=f"{NS}drug", recover=True, huge_tree=True)
for _, elem in context:
    primary = next((x.text.strip() for x in elem.findall(f"{NS}drugbank-id") if x.get("primary") == "true" and x.text), None)
    name = elem.findtext(f"{NS}name")
    if not primary or not name:
        elem.clear(); continue
    selected = primary.lower() in requested or name.strip().lower() in requested
    if selected:
        drugs.append({"drug_id": primary, "drug_name": name.strip()})
        for t in elem.findall(f"{NS}targets/{NS}target"):
            p = t.find(f"{NS}polypeptide")
            if p is not None and p.get("id"): target_rows.append({"drug": primary, "protein": p.get("id").strip()})
        for e in elem.findall(f"{NS}enzymes/{NS}enzyme"):
            p = e.find(f"{NS}polypeptide")
            if p is not None and p.get("id"): enzyme_rows.append({"drug": primary, "protein": p.get("id").strip()})
        for ext in elem.findall(f"{NS}external-identifiers/{NS}external-identifier"):
            if ext.findtext(f"{NS}resource") == "PubChem Compound":
                cid = ext.findtext(f"{NS}identifier")
                if cid and cid.strip().isdigit(): pubchem_rows.append({"drug": primary, "stitch_flat": f"CID1{int(cid.strip()):08d}"})
                break
    # Known positives are collected for every drug, then restricted to the selected set below.
    for interaction in elem.findall(f"{NS}drug-interactions/{NS}drug-interaction"):
        other = interaction.findtext(f"{NS}drugbank-id")
        if other: ddi_rows.append((primary, other.strip()))
    elem.clear()

drug_nodes = pd.DataFrame(drugs).drop_duplicates("drug_id").sort_values("drug_id").reset_index(drop=True)
assert len(drug_nodes) == 100, f"Expected 100 selected drugs; got {len(drug_nodes)}"
selected_ids = set(drug_nodes.drug_id)
target_df = pd.DataFrame(target_rows).drop_duplicates().reset_index(drop=True)
enzyme_df = pd.DataFrame(enzyme_rows).drop_duplicates().reset_index(drop=True)
protein_nodes = pd.DataFrame({"protein_id": sorted(set(target_df.protein) | set(enzyme_df.protein))})

# Canonical undirected known-positive pairs. Do not use validation/test pairs as graph edges later.
positive_id_pairs = sorted({tuple(sorted((a, b))) for a, b in ddi_rows if a in selected_ids and b in selected_ids and a != b})
assert len(positive_id_pairs) == 2792, f"Expected 2792 positive pairs; got {len(positive_id_pairs)}"

drug_nodes.to_csv(OUT / "drug_nodes.csv", index=False)
target_df.to_csv(OUT / "drug_target_edges.csv", index=False)
enzyme_df.to_csv(OUT / "drug_enzyme_edges.csv", index=False)
protein_nodes.to_csv(OUT / "protein_nodes.csv", index=False)
print("drugs", drug_nodes.shape, "positives", len(positive_id_pairs), "targets", target_df.shape, "enzymes", enzyme_df.shape, "proteins", protein_nodes.shape)

# %% 2. Drug features: 2048 Morgan bits + 4 descriptors. Missing structures remain all-zero.
required_ids = set(drug_nodes.drug_id)
smiles = {}
context = etree.iterparse(str(XML_FILE), events=("end",), tag=f"{NS}drug", recover=True, huge_tree=True)
for _, elem in context:
    primary = next((x.text.strip() for x in elem.findall(f"{NS}drugbank-id") if x.get("primary") == "true" and x.text), None)
    if primary in required_ids:
        for prop in elem.findall(f".//{NS}property"):
            if prop.findtext(f"{NS}kind") == "SMILES" and prop.findtext(f"{NS}value"):
                smiles[primary] = prop.findtext(f"{NS}value").strip(); break
    elem.clear()

fp_gen = GetMorganGenerator(radius=2, fpSize=2048)
def features_for(smile):
    if not isinstance(smile, str): return np.zeros(2052, dtype=np.float32)
    mol = Chem.MolFromSmiles(smile)
    if mol is None: return np.zeros(2052, dtype=np.float32)
    fp = np.zeros(2048, dtype=np.float32)
    for b in fp_gen.GetFingerprint(mol).GetOnBits(): fp[b] = 1.0
    desc = np.nan_to_num(np.array([Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol)], dtype=np.float32))
    return np.concatenate([fp, desc])

x_raw = torch.tensor(np.stack([features_for(smiles.get(d)) for d in drug_nodes.drug_id]), dtype=torch.float32)
has_smiles = torch.tensor([d in smiles for d in drug_nodes.drug_id], dtype=torch.bool)
torch.save(x_raw, OUT / "drug_features_2052.pt")
print("feature matrix", tuple(x_raw.shape), "SMILES", int(has_smiles.sum()), "missing", int((~has_smiles).sum()))

# %% 3. STRICT SPLIT BEFORE PU LEARNING (the critical leakage fix)
drug_to_idx = {d:i for i, d in enumerate(drug_nodes.drug_id)}
idx_to_drug = {i:d for d,i in drug_to_idx.items()}
pos_pairs = [(drug_to_idx[a], drug_to_idx[b]) for a,b in positive_id_pairs]

pos_train, pos_hold = train_test_split(pos_pairs, test_size=.30, random_state=SEED)
pos_val, pos_test = train_test_split(pos_hold, test_size=.50, random_state=SEED)
all_pairs = set(combinations(range(len(drug_nodes)), 2))
unknown_pairs = sorted(all_pairs - {tuple(sorted(p)) for p in pos_pairs})
u_train, u_hold = train_test_split(unknown_pairs, test_size=.30, random_state=SEED)
u_val, u_test = train_test_split(u_hold, test_size=.50, random_state=SEED)
print(len(pos_train), len(pos_val), len(pos_test), "positive split;", len(u_train), len(u_val), len(u_test), "unknown split")

# Scale only four continuous descriptors using only training-drug structures; fingerprints stay binary.
train_nodes = sorted({n for p in pos_train for n in p})
x = x_raw.clone()
fit_rows = torch.tensor([i for i in train_nodes if has_smiles[i]], dtype=torch.long)
mean, std = x[fit_rows, 2048:].mean(0), x[fit_rows, 2048:].std(0).clamp_min(1e-6)
x[has_smiles, 2048:] = (x[has_smiles, 2048:] - mean) / std

# %% 4. PU model trains on train positives and train unknowns ONLY
def pair_tensor(pairs): return torch.tensor(pairs, dtype=torch.long)
class TempPU(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.net = nn.Sequential(nn.Linear(dim*2, 256), nn.ReLU(), nn.Dropout(.2), nn.Linear(256, 1))
    def forward(self, feat, pairs):
        a, b = feat[pairs[:,0]], feat[pairs[:,1]]
        return self.net(torch.cat([torch.abs(a-b), a*b], 1)).squeeze(-1)

pu_pairs = pos_train + u_train
pu_y = torch.tensor([1.]*len(pos_train) + [0.]*len(u_train), dtype=torch.float32)
pu_model = TempPU(x.shape[1]).to(device)
pu_ds = TensorDataset(pair_tensor(pu_pairs), pu_y)
pu_loader = DataLoader(pu_ds, batch_size=256, shuffle=True)
pu_loss = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([len(u_train)/len(pos_train)], device=device))
pu_opt = torch.optim.Adam(pu_model.parameters(), lr=1e-3, weight_decay=1e-4)
x_device = x.to(device)
for _ in range(100):
    pu_model.train()
    for p,y in pu_loader:
        loss = pu_loss(pu_model(x_device, p.to(device)), y.to(device)); pu_opt.zero_grad(); loss.backward(); pu_opt.step()
pu_model.eval()
with torch.no_grad(): u_train_scores = torch.sigmoid(pu_model(x_device, pair_tensor(u_train).to(device))).cpu().numpy()
SAFE_THRESHOLD = .30
neg_train = [p for p,s in zip(u_train, u_train_scores) if s < SAFE_THRESHOLD]
if not neg_train: raise RuntimeError("No reliable negatives. Inspect PU score distribution before changing the threshold.")
pd.DataFrame({"drug1":[idx_to_drug[i] for i,j in neg_train], "drug2":[idx_to_drug[j] for i,j in neg_train]}).to_csv(OUT / "reliable_negative_train_pairs.csv", index=False)
print("Reliable training negatives:", len(neg_train))

# Train labels use reliable negatives. Validation/test use untouched unknown pairs as an explicitly named unlabeled-proxy evaluation.
train_pairs, y_train = pos_train + neg_train, np.array([1]*len(pos_train)+[0]*len(neg_train), dtype=np.float32)
val_pairs, y_val = pos_val + u_val, np.array([1]*len(pos_val)+[0]*len(u_val), dtype=np.float32)
test_pairs, y_test = pos_test + u_test, np.array([1]*len(pos_test)+[0]*len(u_test), dtype=np.float32)
train_pt, val_pt, test_pt = pair_tensor(train_pairs).to(device), pair_tensor(val_pairs).to(device), pair_tensor(test_pairs).to(device)
y_train_t, y_val_t = torch.tensor(y_train, device=device), torch.tensor(y_val, device=device)

def bi_edge_index(pairs):
    e=[]
    for i,j in {tuple(sorted(p)) for p in pairs}: e += [[i,j],[j,i]]
    return torch.tensor(e, dtype=torch.long).t().contiguous()
drug_edge_index = bi_edge_index(pos_train)  # TRAIN POSITIVES ONLY
assert not ({tuple(sorted(p)) for p in pos_test} & {tuple(sorted(e)) for e in drug_edge_index.t().tolist()})

# %% 5. Shared training/evaluation functions
def fit_temperature(val_logits, val_labels):
    t = nn.Parameter(torch.ones(1, device=device))
    opt = torch.optim.LBFGS([t], lr=.01, max_iter=100)
    loss_fn = nn.BCEWithLogitsLoss()
    def closure(): opt.zero_grad(); loss=loss_fn(val_logits.detach()/t.clamp_min(.05), val_labels.detach()); loss.backward(); return loss
    opt.step(closure); return t.detach().clamp_min(.05)

def train_link_model(model, graph, tag, epochs=300, patience=50):
    model = model.to(device); graph = graph.to(device)
    with torch.no_grad(): _ = model(graph, train_pt[:1])  # initialize lazy PyG layers
    weight = torch.tensor([float((y_train==0).sum()/(y_train==1).sum())], device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=weight)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    best, best_loss, stale = None, float("inf"), 0
    history=[]
    for epoch in range(epochs):
        model.train(); loss=loss_fn(model(graph, train_pt), y_train_t); opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.); opt.step()
        model.eval()
        with torch.no_grad(): vloss=loss_fn(model(graph, val_pt), y_val_t)
        history.append((float(loss), float(vloss)))
        if vloss < best_loss: best_loss=float(vloss); best=copy.deepcopy(model.state_dict()); stale=0
        else: stale += 1
        if stale >= patience: break
    model.load_state_dict(best); model.eval()
    with torch.no_grad(): val_logits=model(graph,val_pt); test_logits=model(graph,test_pt)
    T=fit_temperature(val_logits, y_val_t); score=torch.sigmoid(test_logits/T).cpu().numpy(); pred=(score>=.5).astype(int)
    metrics={"AUC":roc_auc_score(y_test,score),"Accuracy":accuracy_score(y_test,pred),"Precision":precision_score(y_test,pred,zero_division=0),"Recall":recall_score(y_test,pred,zero_division=0),"F1":f1_score(y_test,pred,zero_division=0),"Brier":brier_score_loss(y_test,score),"best_val_loss":best_loss,"temperature":float(T)}
    print(tag, metrics); return model, graph, metrics, history

# %% 6. Model 0
class Model0(nn.Module):
    def __init__(self, dim, h=128):
        super().__init__(); self.c1=SAGEConv(dim,h); self.c2=SAGEConv(h,h); self.head=nn.Sequential(nn.Linear(2*h,128),nn.ReLU(),nn.Dropout(.3),nn.Linear(128,1))
    def forward(self,data,pairs):
        z=self.c2(F.relu(self.c1(data.x,data.edge_index)),data.edge_index); a,b=z[pairs[:,0]],z[pairs[:,1]]; return self.head(torch.cat([torch.abs(a-b),a*b],1)).squeeze(-1)
data0=Data(x=x,edge_index=drug_edge_index)
model0,data0,m0,h0=train_link_model(Model0(x.shape[1]),data0,"M0")

# %% 7. Download Model 1 biology: STRING high-confidence PPI and KEGG pathways
protein_ids=protein_nodes.protein_id.astype(str).tolist()
STRING="https://string-db.org/api/tsv"
r=requests.post(f"{STRING}/get_string_ids",data={"identifiers":"\r".join(protein_ids),"species":9606,"limit":1,"caller_identity":"ddi_clean_colab"},timeout=120); r.raise_for_status()
mapping=pd.read_csv(StringIO(r.text),sep="\t"); mapping["queryItem"]=mapping.queryIndex.map(lambda i:protein_ids[int(i)])
s2p=dict(zip(mapping.stringId,mapping.queryItem))
r=requests.post(f"{STRING}/network",data={"identifiers":"\r".join(mapping.stringId.unique()),"species":9606,"required_score":700,"add_nodes":0,"caller_identity":"ddi_clean_colab"},timeout=120); r.raise_for_status()
raw_ppi=pd.read_csv(StringIO(r.text),sep="\t")
ppi_df=pd.DataFrame({"protein1":raw_ppi.stringId_A.map(s2p),"protein2":raw_ppi.stringId_B.map(s2p)}).dropna().drop_duplicates()

def batches(a,n):
    for i in range(0,len(a),n): yield a[i:i+n]
u2k={}
for b in batches(protein_ids,10):
    r=requests.get("https://rest.kegg.jp/conv/hsa/"+"+".join("uniprot:"+x for x in b),timeout=60); r.raise_for_status()
    for line in r.text.splitlines():
        a,b2=line.split("\t"); up=next((q[3:] for q in (a,b2) if q.startswith("up:")),None); kg=next((q for q in (a,b2) if q.startswith("hsa:")),None)
        if up and kg: u2k[up]=kg
    time.sleep(.4)
k2u={v:k for k,v in u2k.items()}; path_rows=[]
for b in batches(list(k2u),10):
    r=requests.get("https://rest.kegg.jp/link/pathway/"+"+".join(b),timeout=60); r.raise_for_status()
    for line in r.text.splitlines():
        a,b2=line.split("\t"); kg=next((q for q in (a,b2) if q.startswith("hsa:")),None); pa=next((q.replace("path:","") for q in (a,b2) if q.startswith("path:")),None)
        if kg and pa and kg in k2u: path_rows.append({"protein":k2u[kg],"pathway":pa})
    time.sleep(.4)
pathway_df=pd.DataFrame(path_rows).drop_duplicates()
ppi_df.to_csv(OUT/"string_ppi_edges.csv",index=False); pathway_df.to_csv(OUT/"protein_pathway_edges.csv",index=False)

# %% 8. Generic heterograph builder and Model 1
protein_to_idx={p:i for i,p in enumerate(protein_nodes.protein_id)}
pathway_to_idx={p:i for i,p in enumerate(sorted(pathway_df.pathway.unique()))}
def bip(df,a,b,ai,bi):
    e=[[ai[row[a]],bi[row[b]]] for _,row in df.iterrows() if row[a] in ai and row[b] in bi]
    return torch.tensor(e,dtype=torch.long).t().contiguous()
target_e=bip(target_df,"drug","protein",drug_to_idx,protein_to_idx); enzyme_e=bip(enzyme_df,"drug","protein",drug_to_idx,protein_to_idx)
ppi_e=bi_edge_index([(protein_to_idx[a],protein_to_idx[b]) for a,b in ppi_df[["protein1","protein2"]].itertuples(index=False) if a in protein_to_idx and b in protein_to_idx])
path_e=bip(pathway_df,"protein","pathway",protein_to_idx,pathway_to_idx)

def build_graph(side_df=None,go_df=None):
    d=HeteroData(); d["drug"].x=x; counts={"protein":len(protein_to_idx),"pathway":len(pathway_to_idx)}
    for n,c in counts.items(): d[n].x=torch.arange(c,dtype=torch.long)
    d["drug","interacts","drug"].edge_index=drug_edge_index
    for rel,e in [("targets",target_e),("enzymes",enzyme_e)]: d["drug",rel,"protein"].edge_index=e; d["protein","rev_"+rel,"drug"].edge_index=e.flip(0)
    d["protein","interacts","protein"].edge_index=ppi_e; d["protein","in_pathway","pathway"].edge_index=path_e; d["pathway","rev_in_pathway","protein"].edge_index=path_e.flip(0)
    if side_df is not None:
        side_to_idx={s:i for i,s in enumerate(sorted(side_df.side_effect.unique()))}; go_to_idx={g:i for i,g in enumerate(sorted(go_df.go_term.unique()))}
        d["side_effect"].x=torch.arange(len(side_to_idx),dtype=torch.long); d["go"].x=torch.arange(len(go_to_idx),dtype=torch.long)
        se=bip(side_df,"drug","side_effect",drug_to_idx,side_to_idx); ge=bip(go_df,"protein","go_term",protein_to_idx,go_to_idx)
        d["drug","has_side_effect","side_effect"].edge_index=se; d["side_effect","rev_has_side_effect","drug"].edge_index=se.flip(0)
        d["protein","has_go","go"].edge_index=ge; d["go","rev_has_go","protein"].edge_index=ge.flip(0)
    return d

class HeteroLink(nn.Module):
    def __init__(self, graph, h=128, emb=64, layers=3):
        super().__init__(); self.emb=nn.ModuleDict({n:nn.Embedding(graph[n].num_nodes,emb) for n in graph.node_types if n!="drug"}); self.convs=nn.ModuleList([HeteroConv({r:SAGEConv((-1,-1),h) for r in graph.edge_types},aggr="sum") for _ in range(layers)]); self.head=nn.Sequential(nn.Linear(2*h,128),nn.ReLU(),nn.Dropout(.3),nn.Linear(128,1))
    def forward(self,d,pairs):
        z={"drug":d["drug"].x}; z.update({n:self.emb[n](d[n].x) for n in self.emb})
        for conv in self.convs: z={n:F.relu(v) * (1.4 if n=="side_effect" else 1.) for n,v in conv(z,d.edge_index_dict).items()}
        a,b=z["drug"][pairs[:,0]],z["drug"][pairs[:,1]]; return self.head(torch.cat([torch.abs(a-b),a*b],1)).squeeze(-1)

data1=build_graph(); model1,data1,m1,h1=train_link_model(HeteroLink(data1),data1,"M1")

# %% 9. Model 2 data: SIDER via PubChem/STITCH and GO via UniProt
sider=pd.read_csv("https://sideeffects.embl.de/media/download/meddra_all_se.tsv.gz",sep="\t",header=None,compression="gzip",dtype=str,usecols=[0,5],names=["stitch_flat","side_effect"])
pubchem_df=pd.DataFrame(pubchem_rows).drop_duplicates()
side_df=pubchem_df.merge(sider,on="stitch_flat",how="inner")[["drug","side_effect"]].drop_duplicates()
go_rows=[]
for b in batches(protein_ids,50):
    q=" OR ".join("accession:"+p for p in b); r=requests.get("https://rest.uniprot.org/uniprotkb/search",params={"query":"("+q+")","format":"tsv","fields":"accession,go_id","size":500},timeout=120); r.raise_for_status(); t=pd.read_csv(StringIO(r.text),sep="\t"); gc=next(c for c in t.columns if "Gene Ontology" in c)
    for _,row in t.iterrows():
        for g in str(row[gc]).split("; "):
            if g.startswith("GO:"): go_rows.append({"protein":row["Entry"],"go_term":g})
go_df=pd.DataFrame(go_rows).drop_duplicates()
side_df.to_csv(OUT/"drug_side_effect_edges.csv",index=False); go_df.to_csv(OUT/"protein_go_edges.csv",index=False)
print("SIDER",side_df.shape,"GO",go_df.shape)
data2=build_graph(side_df,go_df); model2,data2,m2,h2=train_link_model(HeteroLink(data2),data2,"M2")

# %% 10. Final, single comparison. These are unlabeled-proxy test metrics, not clinical validation.
comparison=pd.DataFrame([m0,m1,m2],index=["Model 0","Model 1","Model 2"])
comparison.to_csv(OUT/"model_comparison_unlabeled_proxy.csv")
print(comparison[["AUC","Accuracy","Precision","Recall","F1","Brier"]])
print("Saved clean outputs in",OUT)


Mounted at /content/drive
Device: cuda
drugs (100, 2) positives 2792 targets (309, 2) enzymes (297, 2) proteins (216, 1)
feature matrix (100, 2052) SMILES 87 missing 13
1954 419 419 positive split; 1510 324 324 unknown split
Reliable training negatives: 1350


/tmp/ipykernel_890/155989807.py:210: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  history.append((float(loss), float(vloss)))


M0 {'AUC': np.float64(0.9317083591148825), 'Accuracy': 0.8532974427994616, 'Precision': 0.8954081632653061, 'Recall': 0.837708830548926, 'F1': 0.8655980271270037, 'Brier': np.float64(0.10403057711797022), 'best_val_loss': 0.30099985003471375, 'temperature': 1.2129673957824707}
M1 {'AUC': np.float64(0.9550885412062818), 'Accuracy': 0.8815612382234186, 'Precision': 0.9321148825065274, 'Recall': 0.8520286396181385, 'F1': 0.8902743142144638, 'Brier': np.float64(0.0899612815005918), 'best_val_loss': 0.2503512501716614, 'temperature': 1.3354696035385132}
SIDER (6415, 2) GO (6552, 2)
M2 {'AUC': np.float64(0.9427207637231504), 'Accuracy': 0.8707940780619112, 'Precision': 0.9088607594936708, 'Recall': 0.8568019093078759, 'F1': 0.8820638820638821, 'Brier': np.float64(0.09480725847379966), 'best_val_loss': 0.27223557233810425, 'temperature': 1.1604355573654175}
              AUC  Accuracy  Precision    Recall        F1     Brier
Model 0  0.931708  0.853297   0.895408  0.837709  0.865598  0.104031

In [3]:
def risk_level(score):
    if score < 0.4:
        return "LOW"
    elif score > 0.8:
        return "HIGH"
    return "MODERATE"

name_to_idx = {
    name.strip().lower(): drug_to_idx[drug_id]
    for drug_id, name in zip(drug_nodes["drug_id"], drug_nodes["drug_name"])
}

def predict_ddi(drug_name_1, drug_name_2):
    i = name_to_idx[drug_name_1.strip().lower()]
    j = name_to_idx[drug_name_2.strip().lower()]

    pair = torch.tensor([[i, j]], dtype=torch.long, device=device)

    model2.eval()
    with torch.no_grad():
        logit = model2(data2, pair)

        # Use Model 2 validation-fitted temperature
        score = torch.sigmoid(logit / m2["temperature"]).item()

    return {
        "drug1": drug_name_1,
        "drug2": drug_name_2,
        "predicted_ddi_probability": round(score, 4),
        "risk": risk_level(score)
    }

# Example
predict_ddi("Metformin", "Amlodipine")

{'drug1': 'Metformin',
 'drug2': 'Amlodipine',
 'predicted_ddi_probability': 0.6871,
 'risk': 'MODERATE'}

In [6]:
# =========================================================
# RANK THE 813 REMAINING UNKNOWN DRUG PAIRS
# =========================================================

# Unknown training pairs that were NOT selected as reliable negatives
unselected_train_unknown = set(u_train) - set(neg_train)

# Add held-out validation and test unknown pairs
remaining_unknown_pairs = sorted(
    unselected_train_unknown | set(u_val) | set(u_test)
)

print("Remaining unknown pairs:", len(remaining_unknown_pairs))

pair_tensor = torch.tensor(
    remaining_unknown_pairs,
    dtype=torch.long,
    device=device
)

model2.eval()

with torch.no_grad():
    logits = model2(data2, pair_tensor)
    scores = torch.sigmoid(
        logits / m2["temperature"]
    ).cpu().numpy()

id_to_name = dict(zip(drug_nodes["drug_id"], drug_nodes["drug_name"]))

def risk_level(score):
    if score < 0.4:
        return "LOW"
    elif score > 0.8:
        return "HIGH"
    return "MODERATE"

unknown_predictions = pd.DataFrame([
    {
        "drug1": id_to_name[idx_to_drug[i]],
        "drug2": id_to_name[idx_to_drug[j]],
        "predicted_ddi_probability": float(score),
        "risk": risk_level(float(score))
    }
    for (i, j), score in zip(remaining_unknown_pairs, scores)
])

unknown_predictions = unknown_predictions.sort_values(
    "predicted_ddi_probability",
    ascending=False
).reset_index(drop=True)

unknown_predictions.to_csv(
    OUT / "remaining_unknown_pair_predictions.csv",
    index=False
)

print(unknown_predictions["risk"].value_counts())
display(unknown_predictions.head(20))

Remaining unknown pairs: 808
risk
LOW         632
MODERATE    141
HIGH         35
Name: count, dtype: int64


,drug1,drug2,predicted_ddi_probability,risk
0,Metoprolol,Felodipine,0.999934,HIGH
1,Metoprolol,Nifedipine,0.999516,HIGH
2,Glimepiride,Rosiglitazone,0.998817,HIGH
3,Glimepiride,Pioglitazone,0.989837,HIGH
4,Enalapril,Nitrendipine,0.985294,HIGH
5,Rosiglitazone,Warfarin,0.976251,HIGH
6,Enalapril,Felodipine,0.975113,HIGH
7,Linagliptin,Empagliflozin,0.950614,HIGH
8,Valsartan,Amlodipine,0.936249,HIGH
9,Clopidogrel,Acetylsalicylic acid,0.933409,HIGH


In [11]:
# =========================================================
# SAVE FINAL MODELS AND REPRODUCIBILITY ARTIFACTS
# =========================================================

import json

# Model checkpoints
torch.save(model0.state_dict(), OUT / "model0_state_dict.pt")
torch.save(model1.state_dict(), OUT / "model1_state_dict.pt")
torch.save(model2.state_dict(), OUT / "model2_state_dict.pt")

# Drug-index mapping used by every model
with open(OUT / "drug_index_mapping.json", "w") as f:
    json.dump(
        {str(index): drug_id for index, drug_id in idx_to_drug.items()},
        f,
        indent=2
    )

# Save final metrics
comparison.to_csv(OUT / "final_model_metrics.csv")

# Save reproducibility settings
settings = {
    "seed": SEED,
    "num_drugs": 100,
    "known_positive_pairs": 2792,
    "unknown_pairs": 2158,
    "training_positive_pairs": len(pos_train),
    "reliable_training_negatives": len(neg_train),
    "safe_threshold": SAFE_THRESHOLD,
    "string_score_threshold": 700,
    "remaining_unknown_pairs_ranked": len(remaining_unknown_pairs),
    "evaluation_note": "Unknown-pair proxy evaluation; not clinical validation."
}

with open(OUT / "experiment_settings.json", "w") as f:
    json.dump(settings, f, indent=2)

print("Saved final artifacts to:", OUT)
print(sorted(path.name for path in OUT.iterdir()))

Saved final artifacts to: /content/drive/MyDrive/GNN_PROJECT/outputs_clean
['drug_enzyme_edges.csv', 'drug_features_2052.pt', 'drug_index_mapping.json', 'drug_nodes.csv', 'drug_side_effect_edges.csv', 'drug_target_edges.csv', 'experiment_settings.json', 'final_model_metrics.csv', 'model0_state_dict.pt', 'model1_state_dict.pt', 'model2_state_dict.pt', 'model_comparison_unlabeled_proxy.csv', 'protein_go_edges.csv', 'protein_nodes.csv', 'protein_pathway_edges.csv', 'reliable_negative_train_pairs.csv', 'remaining_813_unknown_pair_predictions.csv', 'remaining_unknown_pair_predictions.csv', 'string_ppi_edges.csv']


In [12]:
# =========================================================
# POLYPHARMACY RISK ANALYSIS
# =========================================================

from itertools import combinations

name_lookup = {
    name.strip().lower(): (
        drug_id,
        drug_to_idx[drug_id]
    )
    for drug_id, name in zip(
        drug_nodes["drug_id"],
        drug_nodes["drug_name"]
    )
}

known_positive_set = {
    tuple(sorted(pair))
    for pair in pos_pairs
}

def risk_level(score):
    if score < 0.4:
        return "LOW"
    elif score > 0.8:
        return "HIGH"
    return "MODERATE"

def analyze_polypharmacy(drug_names):
    if len(drug_names) < 2:
        raise ValueError("Enter at least two drugs.")

    resolved = []
    missing = []

    for name in drug_names:
        item = name_lookup.get(name.strip().lower())

        if item is None:
            missing.append(name)
        else:
            resolved.append((name, item[0], item[1]))

    if missing:
        raise ValueError(f"Drug names not found in selected 100 drugs: {missing}")

    indices = [item[2] for item in resolved]
    pair_indices = list(combinations(indices, 2))

    pair_tensor = torch.tensor(
        pair_indices,
        dtype=torch.long,
        device=device
    )

    model2.eval()

    with torch.no_grad():
        logits = model2(data2, pair_tensor)
        scores = torch.sigmoid(
            logits / m2["temperature"]
        ).cpu().numpy()

    rows = []

    for (i, j), score in zip(pair_indices, scores):
        pair = tuple(sorted((i, j)))
        known = pair in known_positive_set

        rows.append({
            "drug1": id_to_name[idx_to_drug[i]],
            "drug2": id_to_name[idx_to_drug[j]],
            "model_score": float(score),
            "result": (
                "KNOWN_DRUGBANK_DDI"
                if known
                else f"MODEL_ESTIMATED_{risk_level(float(score))}_RISK"
            ),
            "known_drugbank_ddi": known
        })

    pair_results = pd.DataFrame(rows).sort_values(
        "model_score",
        ascending=False
    ).reset_index(drop=True)

    high_risk_pairs_df = pair_results[
        (pair_results["known_drugbank_ddi"]) |
        (pair_results["result"] == "MODEL_ESTIMATED_HIGH_RISK")
    ]

    problem_drugs = sorted(set(
        high_risk_pairs_df["drug1"].tolist() +
        high_risk_pairs_df["drug2"].tolist()
    ))

    # SOP heuristic: penalizes multiple high-risk pair scores
    total_regimen_risk = float(
        np.sum(pair_results["model_score"] ** 2)
    )

    return {
        "pair_results": pair_results,
        "high_risk_pairs": high_risk_pairs_df,
        "problem_drugs": problem_drugs,
        "total_regimen_risk": total_regimen_risk
    }

# Example: replace with drugs available in your selected 100-drug list
regimen = ["Metformin", "Glimepiride", "Amlodipine", "Atorvastatin"]

result = analyze_polypharmacy(regimen)

display(result["pair_results"])
print("Problem drugs:", result["problem_drugs"])
print("Total regimen risk:", result["total_regimen_risk"])

,drug1,drug2,model_score,result,known_drugbank_ddi
0,Glimepiride,Amlodipine,0.978280,KNOWN_DRUGBANK_DDI,True
1,Metformin,Amlodipine,0.687122,KNOWN_DRUGBANK_DDI,True
2,Metformin,Glimepiride,0.539907,MODEL_ESTIMATED_MODERATE_RISK,False
3,Glimepiride,Atorvastatin,0.438219,KNOWN_DRUGBANK_DDI,True
4,Amlodipine,Atorvastatin,0.341234,MODEL_ESTIMATED_LOW_RISK,False
5,Metformin,Atorvastatin,0.293187,KNOWN_DRUGBANK_DDI,True


Problem drugs: ['Amlodipine', 'Atorvastatin', 'Glimepiride', 'Metformin']
Total regimen risk: 2.115103795417026
